# Conditioning

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/linear_systems/conditioning.ipynb)

In [4]:
# | tags: [remove-cell]
try:
    import executable_engineering as exe 
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

import numpy as np

## Stability and Uncertainty

Previously, we discussed how roundoff and truncation errors introduce uncertainty into numbers stored on a computer. The **conditioning** of a function relates how uncertainty in the inputs propagates to uncertainty in the outputs. 

For a general function $f(\mathbf{x}) = \mathbf{b}$:
* **Well-conditioned:** Small changes in inputs ($\mathbf{x}$) lead to proportionally small changes in outputs ($\mathbf{b}$).
* **Ill-conditioned:** Small changes in inputs *could* result in massive, explosive changes in the outputs, and vice-versa.

The condition of a function is a fundamental property of the mathematical transformation itself, not just the specific input/output values.

Recall that numerical methods have inherent uncertainty in numbers due to finite precision representation. Ill-conditioned functions can catastrophically disrupt finding solutions of linear systems. 

## Distortion, not Amplification

A common misconception is that ill-conditioning is just "amplification". But if a function simply amplified error, its *inverse* would reduce error.

This is not the case! The conditioning of a forward function and its inverse are **quantifiably the same**. Conditioning represents the maximum *distortion* or *stretching* caused by the transformation in any direction.

## Visualizing Matrix Distortion

For a linear transformation $\mathbf{A}\mathbf{x} = \mathbf{b}$, we can visualize this distortion by applying the matrix $\mathbf{A}$ (and its inverse $\mathbf{A}^{-1}$) to a perfect unit circle.

Let's look at a **well-conditioned** matrix:
$$
\mathbf{A} = \begin{pmatrix} 2 & 0 \\ 0 & 2 \end{pmatrix}
$$

In [5]:
A_well = [[2, 0], 
          [0, 2]]

fig1 = exe.visualize_conditioning(A_well)
fig1.show()

Notice how the circle is scaled but remains perfectly proportional. The inverse simply shrinks it back.

Now, let's look at an **ill-conditioned** matrix where the rows are nearly parallel:
$$
\mathbf{A} = \begin{pmatrix} 1 & 1 \\ 1 & 1.1 \end{pmatrix}
$$

In [6]:
A_ill = [[1, 1], 
         [1, 1.1]]

fig2 = exe.visualize_conditioning(A_ill)
fig2.show()

## The Condition Number ($\kappa$)

The severe stretching seen above is quantified by the **Condition Number** ($\kappa$). It is the maximum ratio of the relative error in $\mathbf{x}$ to the relative error in $\mathbf{b}$.

Let's perturb our inputs and outputs by a small error $\delta \mathbf{x}$ and $\delta \mathbf{b}$:

$$
\begin{aligned}
\mathbf{x} & \rightarrow \mathbf{x} + \delta \mathbf{x}\\
\mathbf{b} & \rightarrow \mathbf{b} + \delta \mathbf{b}
\end{aligned}
$$

### Derivation

Substitute the perturbed vectors into our linear system $\mathbf{A} \mathbf{x} = \mathbf{b}$:

$$
\begin{aligned}
\mathbf{A} (\mathbf{x} + \delta \mathbf{x}) &= \mathbf{b} + \delta \mathbf{b} \\
\mathbf{A} \mathbf{x} + \mathbf{A} \delta \mathbf{x} &= \mathbf{b} + \delta \mathbf{b}
\end{aligned}
$$

Since we know $\mathbf{A} \mathbf{x} = \mathbf{b}$, these terms cancel out, leaving us with a direct relationship between the uncertainties:

$$
\begin{aligned}
\mathbf{A} \delta \mathbf{x} &= \delta \mathbf{b} \\
\delta \mathbf{x} &= \mathbf{A}^{-1} \delta \mathbf{b}
\end{aligned}
$$

### Ratio of Relative Errors

We want to find the ratio of the relative error in the input to the relative error in the output using the **norm** (magnitude) of the vectors:

$$
\frac{\|\delta \mathbf{x}\|}{\|\mathbf{x}\|} \Bigg/ \frac{\|\delta \mathbf{b}\|}{\|\mathbf{b}\|} = \frac{\|\mathbf{A}^{-1}\delta \mathbf{b}\|}{\|\delta \mathbf{b}\|} \cdot \frac{\|\mathbf{A} \mathbf{x}\|}{\|\mathbf{x}\|}
$$

The maximum possible value of this ratio across all possible vectors is defined as the condition number $\kappa$:

$$
\kappa = \max \left( \frac{\|\mathbf{A}^{-1}\delta \mathbf{b}\|}{\|\delta \mathbf{b}\|} \right) \cdot \max \left( \frac{\|\mathbf{A} \mathbf{x}\|}{\|\mathbf{x}\|} \right)
$$

$$
\kappa = \|\mathbf{A}^{-1}\| \|\mathbf{A}\|
$$

> NB: This result introduces the concept of the norm (magnitude) of a matrix (defined below) and an identity $\|\mathbf{A}^{-1}\delta \mathbf{b}\| \leq \|\mathbf{A}^{-1} \| \cdot \|\delta \mathbf{b}\|$. 

### Application

From this definition, it is obvious that the condition number is identical for both the forward ($\mathbf{A}$) and backward ($\mathbf{A}^{-1}$) transformations!

It guarantees bounds on the relative error in both directions:

$$
\frac{\|\delta \mathbf{b}\|}{\|\mathbf{b}\|} \leq \kappa \frac{\|\delta \mathbf{x}\|}{\|\mathbf{x}\|} \quad \text{and} \quad \frac{\|\delta \mathbf{x}\|}{\|\mathbf{x}\|} \leq \kappa \frac{\|\delta \mathbf{b}\|}{\|\mathbf{b}\|}
$$

If $\kappa$ is large (ill-conditioned), small errors in your data $\mathbf{b}$ can result in massive, unpredictable errors in your computed solution $\mathbf{x}$ and due to finite precision, small errors will *always* be present!!

## Calculation via Eigenvalues

The formal definition requires calculating the inverse $\mathbf{A}^{-1}$, which is exactly what we are trying to avoid doing blindly! We want to know the condition number *before* attempting to solve the system.

Geometrically, the maximal stretching of a matrix is defined by its largest eigenvalue ($\lambda_{\max}$). The maximal stretching of the inverse is the smallest eigenvalue of the original matrix ($\lambda_{\min}$). 

Therefore, for symmetric matrices, we can calculate $\kappa$ using the eigenvalues:

$$
\kappa = \frac{|\lambda_{\max}|}{|\lambda_{\min}|}
$$

## Matrix Norms

For vectors, the standard Euclidean magnitude (or $L^2$ norm) is:
$$ \|\mathbf{v}\|_2 = \sqrt{\sum v_i^2} $$

For matrices, the equivalent Frobenius norm is the square root of the sum of the absolute squares of its elements:
$$ \|\mathbf{A}\|_F = \sqrt{\sum_{i=1}^{m} \sum_{j=1}^{n} |A_{ij}|^2} $$

The **Infinity Norm** (maximum row-sum norm) is often used for faster computation:
$$ \|\mathbf{A}\|_\infty = \max_{1 \le i \le m} \sum_{j=1}^{n} |A_{ij}| $$